[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/03_comparison.ipynb)


# R1-Q2 Week 4 — Compare hubs to known regulators

You arrive at this notebook with two per-tissue hub lists (212 shoot
hubs, 467 root hubs) from Notebook 2. The question this notebook
answers is: **are those hubs enriched for known stress-regulating
transcription factors?**

The reference set is Hakkak & Tohidfar (2026) Supplementary Table 3 —
roughly 60 TFs across 15 families, all identified from a stress-response
meta-analysis. If a co-expression network really does surface
biologically meaningful regulators, the hubs and the reference should
overlap more than chance would predict.

By the end of this notebook you will have:

- Per-tissue hub tables annotated against the Hakkak reference, saved
  as `shoot_compare.parquet` and `root_compare.parquet`.
- A formal enrichment test (one-sided hypergeometric, Bonferroni-
  corrected) for each tissue.
- A descriptive look at the 66 cross-tissue hubs against the reference.
- A summary JSON capturing the results for use in the writeup.
- The answer to R1-Q2's question, in your own words, written as the
  Section 6 synthesis.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Setup and load inputs

Three pieces of input feed the rest of this notebook:

1. **The per-tissue hub tables from N2** (`shoot_hubs.parquet`,
   `root_hubs.parquet`). Each row is a probe identified as a hub in
   a stress-relevant module.
2. **The reference set** (`hakkak_2026_supp3.csv`) — a curated list
   of stress-responsive transcription factors from Hakkak & Tohidfar
   (2026). The file is uploaded to the same R1-Q2 output directory
   that N2 wrote to; if you haven't put it there yet, see the load
   cell below for download instructions.
3. **The pre-commits** (`precommit.json`) — your Week 1 lock-ins for
   the reference set, the statistical framework, and the background
   gene set. We echo the three entries that this notebook acts on so
   the locked decisions are visible while you read the analysis.

Each load below has a defensive error path. If a file isn't found,
the error tells you what's missing and how to produce it.

### 1.1) Hub tables from N2

Two Parquet files, one per tissue. Each row is a probe (the index),
with columns capturing module membership, the probe's kME value
within that module, and the stresses the module is associated with.

In [ ]:
import pandas as pd

shoot_hubs_path = OUTPUT_DIR / "shoot_hubs.parquet"
root_hubs_path = OUTPUT_DIR / "root_hubs.parquet"

try:
    shoot_hubs = pd.read_parquet(shoot_hubs_path)
    root_hubs = pd.read_parquet(root_hubs_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find an N2 hub table.\n"
        f"  Looked for: {shoot_hubs_path}\n"
        f"              {root_hubs_path}\n\n"
        f"Run `02_hub_identification.ipynb` to completion first — "
        f"its Section 5 writes both Parquets to this location.\n"
    ) from None

print(f"Shoot hubs: {len(shoot_hubs):>4} probes")
print(f"Root hubs:  {len(root_hubs):>4} probes")
print()
print(f"Columns:     {shoot_hubs.columns.tolist()}")
print(f"Index name:  {shoot_hubs.index.name!r}")
print()
print("Shoot head:")
print(shoot_hubs.head())

### 1.2) Reference set (Hakkak & Tohidfar 2026 Supplementary Table 3)

The reference file is a curated list of transcription factors that
Hakkak & Tohidfar identified as differentially expressed across a
combined drought/salt meta-analysis. Each row records:

- `ORF` — the AGI gene identifier (the column is called `ORF` in the
  source file, not `agi` or `gene_id`; this is a Hakkak convention).
- `Symbol` — the canonical gene symbol (e.g., `DREB2A`).
- `Family` — the transcription factor family (e.g., `ERF`, `MYB`).
- `Group` — `Up-regulated gene` or `Down-regulated gene`.

The source file has a layout quirk: it's published as a
two-column-block CSV, with up-regulated TFs in the left block and
down-regulated TFs in the right block, separated by a blank column.
The load cell collapses these into one tidy table.

Two more quirks to be aware of, both handled below:

1. **The `ORF` column is in mixed case** (`At1g01720` rather than
   `AT1G01720`). The probe-to-AGI mapping in Section 2 normalizes
   to upper case, so we apply the same normalization here at load
   time. Otherwise joins downstream would silently miss everything.
2. **One row is duplicated.** `AT2G47190` appears twice — once with
   family `ERF`, once with family `MYB`. Same gene, conflicting
   family label in the source. The defensive check below surfaces
   any duplicates so you can see them; we carry both rows in the
   loaded table for fidelity to source, and handle the dedup
   downstream where it matters (Section 3, the per-family
   breakdown).

If the file isn't present, download it from the journal's
supplementary materials. Springer publishes it as
`10709_2026_261_MOESM1_ESM.csv` (despite our internal name being
`...supp3...` — the file numbering on the journal site doesn't match
the paper's narrative table numbering). Rename and upload to
`OUTPUT_DIR`.

In [ ]:
hakkak_path = OUTPUT_DIR / "hakkak_2026_supp3.csv"

try:
    raw = pd.read_csv(hakkak_path, skiprows=1)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find the Hakkak reference set at:\n"
        f"  {hakkak_path}\n\n"
        f"Download `10709_2026_261_MOESM1_ESM.csv` from the journal's "
        f"supplementary materials for Hakkak & Tohidfar (2026), rename "
        f"it to `hakkak_2026_supp3.csv`, and place it at the path above.\n"
    ) from None

# The file's two-column-block layout: columns 0-3 are up-regulated TFs,
# column 4 is a blank separator, columns 5-8 are down-regulated TFs.
left = raw.iloc[:, [0, 1, 2, 3]].copy()
right = raw.iloc[:, [5, 6, 7, 8]].copy()
right.columns = left.columns

hakkak_tf = (
    pd.concat([left, right], ignore_index=True)
    .dropna(how='all')
    .reset_index(drop=True)
)

# Normalize case on the AGI column; strip any stray whitespace.
hakkak_tf['ORF'] = hakkak_tf['ORF'].astype(str).str.strip().str.upper()

# Defensive: drop any non-AGI rows (e.g., trailing footer rows).
hakkak_tf = hakkak_tf[hakkak_tf['ORF'].str.startswith('AT')].reset_index(drop=True)

print(f"Reference rows:    {len(hakkak_tf)}")
print(f"Unique AGIs:       {hakkak_tf['ORF'].nunique()}")
print(f"Unique families:   {hakkak_tf['Family'].nunique()}")
print()
print("Family breakdown:")
print(hakkak_tf['Family'].value_counts().to_string())

In [ ]:
# The source file is known to carry one duplicate AGI with conflicting
# family labels. Print any duplicates so the student sees them; we
# carry the rows in `hakkak_tf` and handle the dedup downstream where
# it matters.
duplicates = (
    hakkak_tf[hakkak_tf['ORF'].duplicated(keep=False)]
    .sort_values('ORF')
)

if len(duplicates) > 0:
    print(f"Duplicate AGIs in the reference set ({len(duplicates)} rows):")
    print(duplicates.to_string(index=False))
    print()
    print("Both rows are carried in `hakkak_tf` for fidelity to source.")
    print("Section 3's per-family breakdown deduplicates by uppercase ORF.")
else:
    print("No duplicate AGIs in the reference set.")

### 1.3) Pre-commit echo

Three of the six pre-commits you locked in N0 inform what this
notebook does:

- `reference_set` — which file to compare against, and which column
  holds the AGI codes.
- `statistical_framework` — which test to run, with how many tests
  and what correction.
- `background_gene_set` — the universe of probes that could have
  been hubs, per tissue. Section 4's hypergeometric test uses these
  counts as its `N` (population size) parameter.

The cell below loads `precommit.json` and prints these three blocks
in full. Cross-check the values against what you remember locking in
Week 1; if anything looks wrong, stop and re-open N0 before running
the rest of the notebook.

In [ ]:
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path) as f:
    precommit = json.load(f)

for category in ("reference_set", "statistical_framework", "background_gene_set"):
    print(f"--- {category} ---")
    print(json.dumps(precommit[category], indent=2))
    print()

With the three inputs loaded and the locked decisions in view, the
foreground (hubs) and reference (Hakkak's TF table) are in hand —
but in different identifier spaces. The hub tables are keyed by
Affymetrix probe ID (`249264_s_at`); Hakkak's table is keyed by AGI
gene identifier (`AT5G42570`). Section 2 bridges that gap with a
single library call.

## 2) Map probes to AGI

The hub tables and backgrounds are keyed by Affymetrix ATH1 probe ID
(e.g., `249264_s_at`); Hakkak's reference is keyed by AGI gene
identifier (e.g., `AT5G42570`). To compare them, we translate the
probes to AGIs.

The translation uses `iri.probe_to_agi()` — a library function that
fetches the GPL198 annotation table from GEO (cached locally after
the first call), reads the probe-to-AGI mapping, and returns a
dictionary. Two implementation rules are built into the function and
match what you locked in Week 1:

1. **Multi-locus probes** (probes that target more than one gene)
   get their first locus as the primary mapping. A small fraction
   of the array.
2. **Probes without an AGI annotation** are dropped. This includes
   the Affymetrix control probes (the `AFFX-` prefixed set) and a
   small number of design-stage probes never matched to a locus.

After the mapping, we apply two more pre-commit rules that turn the
probe-keyed data into the AGI-keyed universe the enrichment test
needs:

- **Pre-commit (universe):** drop any probe with no AGI mapping.
  Apply this consistently to both the hub set (the numerator) and
  the background set (the denominator). Probes that can't be
  evaluated against an AGI-keyed reference don't belong in either.
- **Pre-commit (multi-mapping):** when several probes map to the
  same AGI, the AGI counts as a hub if any of its probes is in the
  hub set. Using Python sets makes this collapse happen naturally —
  set deduplication does the work.

The output of this section is five AGI sets:

- `shoot_hubs_agi`, `root_hubs_agi` — the foreground (hubs).
- `shoot_background_agi`, `root_background_agi` — the universe of
  probes that could have been hubs (N0's MAD-filtered probes).
- `hakkak_agi` — the reference set, as a set rather than a
  DataFrame, for the set-algebra steps in Section 3.

### 2.1) Load the backgrounds

Section 4's hypergeometric test needs a universe of probes that
*could* have been hubs — every probe that survived N0's variance
filter and was therefore eligible to land in a co-expression module.
That set lives in the two filtered expression matrices N0 wrote:

- `shoot_filtered.parquet` — about 5,700 probes × 124 samples.
- `root_filtered.parquet`  — about 5,700 probes × 124 samples.

We only need the probe index from these files, not the expression
values, so we load and immediately extract the index.

In [ ]:
shoot_filtered_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_filtered_path = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot_background_probes = pd.read_parquet(shoot_filtered_path).index
    root_background_probes = pd.read_parquet(root_filtered_path).index
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find an N0 filtered matrix.\n"
        f"  Looked for: {shoot_filtered_path}\n"
        f"              {root_filtered_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its Section 3 writes both Parquets to this location.\n"
    ) from None

print(f"Shoot background: {len(shoot_background_probes):>5,} probes")
print(f"Root background:  {len(root_background_probes):>5,} probes")

### 2.2) Build the probe→AGI mapping

One library call. The first run fetches the GPL198 annotation table
from GEO (a few seconds on a typical connection); subsequent runs in
the same session read from a local cache.

The spot-check below confirms the mapping is working: a known
cold-stress regulator (CBF1) should map to its TAIR locus, and an
Affymetrix control probe should map to nothing (dropped by the
function's no-AGI rule).

In [ ]:
probe_to_agi = iri.probe_to_agi()

print(f"Probes with AGI mapping: {len(probe_to_agi):,}")
print()
print("Spot-checks:")
print(f"  CBF1 (254074_at)        -> {probe_to_agi.get('254074_at')!r}")
print(f"  N2 standout (249264_s_at) -> {probe_to_agi.get('249264_s_at')!r}")
print(f"  AFFX-BioB-3_at (control)  -> {probe_to_agi.get('AFFX-BioB-3_at')!r}")

### 2.3) Translate to AGI sets

Apply the mapping to all four probe lists (two hubs, two
backgrounds), drop probes with no AGI mapping, and collapse to
unique AGIs by converting to Python sets. Convert Hakkak's `ORF`
column to a set in the same step so all five AGI universes have
the same type going into Section 3.

The report cell at the end of the translation surfaces three counts
per set:

- **probes in:** how many probes started in the list.
- **unmapped (dropped):** probes with no AGI in GPL198 (control
  probes, design-stage probes). These get dropped per the pre-commit.
- **AGIs out:** the size of the final AGI set. The difference between
  (probes in − unmapped) and (AGIs out) is the number of
  multi-probe-to-same-AGI collapses.

The closing assertion confirms the hubs are a subset of their
respective backgrounds. If they aren't, N0 and N2 disagree on the
eligible probe set and Section 4's test would produce meaningless
numbers — better to fail loudly here than silently downstream.

In [ ]:
def _to_agi_set(probe_iterable):
    """
    Translate an iterable of probe IDs to a set of AGIs.

    Drops probes with no AGI mapping (per the universe pre-commit).
    Multiple probes mapping to the same AGI collapse to one entry
    via set deduplication (per the multi-mapping pre-commit).
    """
    return {agi for probe in probe_iterable
            if (agi := probe_to_agi.get(probe)) is not None}


# Foreground (hubs) and background (eligible universe) for each tissue.
shoot_hubs_agi       = _to_agi_set(shoot_hubs.index)
root_hubs_agi        = _to_agi_set(root_hubs.index)
shoot_background_agi = _to_agi_set(shoot_background_probes)
root_background_agi  = _to_agi_set(root_background_probes)

# Reference set, already in AGI space — just convert to set.
hakkak_agi = set(hakkak_tf['ORF'])


def _report(label, probes, agi_set):
    n_probes = len(probes)
    n_mapped = sum(1 for p in probes if p in probe_to_agi)
    n_unmapped = n_probes - n_mapped
    n_collapsed = n_mapped - len(agi_set)
    print(f"{label}:")
    print(f"  probes in:          {n_probes:>6,}")
    print(f"  unmapped (dropped): {n_unmapped:>6,}")
    print(f"  AGIs out:           {len(agi_set):>6,}")
    if n_collapsed > 0:
        print(f"  (multi-probe→AGI collapses: {n_collapsed})")
    print()

_report("shoot hubs",       shoot_hubs.index,        shoot_hubs_agi)
_report("root hubs",        root_hubs.index,         root_hubs_agi)
_report("shoot background", shoot_background_probes, shoot_background_agi)
_report("root background",  root_background_probes,  root_background_agi)

print(f"hakkak (reference): {len(hakkak_agi)} unique AGIs "
      f"(from {len(hakkak_tf)} rows — duplicates collapsed)")
print()

# Defensive: hubs must be a subset of the background. If they're not,
# something has gone wrong upstream and the test results would be
# meaningless.
assert shoot_hubs_agi.issubset(shoot_background_agi), (
    "Shoot hubs are not a subset of the shoot background. "
    "N0 and N2 disagree on the eligible probe set."
)
assert root_hubs_agi.issubset(root_background_agi), (
    "Root hubs are not a subset of the root background. "
    "N0 and N2 disagree on the eligible probe set."
)
print("Subset check: hubs ⊆ background for both tissues. OK.")

With the five AGI sets in hand, the foreground and reference are now
in the same identifier space and ready to be compared. Section 3
does the join: tag each hub AGI by whether it's in the Hakkak
reference, attach the TF family annotation where applicable, and
produce the per-tissue comparison tables that drive both the
enrichment test (Section 4) and the synthesis (Section 6).

## 3) Compare hubs against the reference

With everything in AGI space, the comparison is a left-join: for each
hub AGI in each tissue, look it up in Hakkak's reference set and
record whether it's there, plus the family and symbol annotations
when it is.

The output is two DataFrames — `shoot_compare` and `root_compare` —
that the rest of the notebook consumes. Each has one row per hub AGI
in that tissue, with columns:

- `agi_id` — the gene identifier.
- `n_hub_probes` — how many of this gene's probes appeared in the
  hub set. Most genes have one probe; multi-probe genes are flagged
  by `n_hub_probes > 1`.
- `in_hakkak` — boolean, True if this AGI is in the reference set.
- `symbol`, `family`, `group` — Hakkak's annotations. `NaN` for
  out-of-reference hubs.

The descriptive numbers — overlap count, top families — surface in
this section. The formal test (is the overlap larger than chance?)
is Section 4.

### 3.1) Resolve the reference duplicate

Section 1 surfaced one duplicate AGI in the reference set:
AT2G47190 is listed twice with conflicting family labels (ERF and
MYB). The gene's canonical symbol is AtMYB2, which makes MYB the
biologically correct label and the ERF entry a known mislabel in
the source.

For the per-AGI join below, we need one row per AGI. We drop the
ERF row explicitly — a content-based filter rather than a
position-based `drop_duplicates(..., keep='first')` call, so the
choice is robust to whatever row order the loaded table happens to
have. The Section 4 hypergeometric test wouldn't care (set
membership doesn't double-count), but the Section 3.4 per-family
breakdown would: without this resolution, ERF would inflate from
its true count of 8 to 9.

In [ ]:
# Drop the ERF-labeled row for AT2G47190. After this, hakkak_tf_dedup
# has one row per unique AGI.
to_drop = (hakkak_tf['ORF'] == 'AT2G47190') & (hakkak_tf['Family'] == 'ERF')
hakkak_tf_dedup = hakkak_tf[~to_drop].reset_index(drop=True)

# Sanity check: no duplicate AGIs should remain.
assert hakkak_tf_dedup['ORF'].nunique() == len(hakkak_tf_dedup), (
    "Unexpected duplicate AGIs remain after resolution."
)

print(f"Reference rows after dedup: {len(hakkak_tf_dedup)} (from {len(hakkak_tf)})")
print()
print("Family counts after dedup:")
print(hakkak_tf_dedup['Family'].value_counts().to_string())

### 3.2) Build per-tissue comparison tables

For each tissue, we collapse the hub table from probe-level to
AGI-level, then left-join against the deduplicated reference. The
result is two parallel DataFrames keyed on `agi_id`.

Within each table, rows are sorted with reference hubs first, then
alphabetical by AGI. The reasoning: the "familiar names" are what
the prediction expected to see, and putting them at the top of the
table makes the next subsection's peek straightforward.

In [ ]:
def _build_compare(hubs_df, probe_to_agi, hakkak_dedup):
    """
    Build a per-AGI comparison table for one tissue.

    Aggregates a probe-level hub table to one row per AGI, then
    left-joins against the (deduplicated) reference. Probes with no
    AGI mapping are dropped per the universe pre-commit.
    """
    # Add AGI column; drop probes with no AGI mapping.
    df = hubs_df.copy()
    df['agi_id'] = df.index.map(probe_to_agi)
    df = df.dropna(subset=['agi_id'])

    # Per-AGI: how many of this gene's probes were in the hub set?
    per_agi = (
        df.groupby('agi_id')
        .size()
        .reset_index(name='n_hub_probes')
    )

    # Left-join Hakkak's annotations.
    annotations = (
        hakkak_dedup
        .rename(columns={
            'ORF': 'agi_id',
            'Symbol': 'symbol',
            'Family': 'family',
            'Group': 'group',
        })
        [['agi_id', 'symbol', 'family', 'group']]
    )
    per_agi = per_agi.merge(annotations, on='agi_id', how='left')
    per_agi['in_hakkak'] = per_agi['family'].notna()

    # Sort: reference hubs first, alphabetical within each group.
    return (
        per_agi
        .sort_values(['in_hakkak', 'agi_id'], ascending=[False, True])
        .reset_index(drop=True)
    )


shoot_compare = _build_compare(shoot_hubs, probe_to_agi, hakkak_tf_dedup)
root_compare  = _build_compare(root_hubs,  probe_to_agi, hakkak_tf_dedup)

print(f"Shoot: {len(shoot_compare)} unique hub AGIs")
print(f"  reference hubs:     {shoot_compare['in_hakkak'].sum():>3}")
print(f"  out-of-reference:   {(~shoot_compare['in_hakkak']).sum():>3}")
print()
print(f"Root:  {len(root_compare)} unique hub AGIs")
print(f"  reference hubs:     {root_compare['in_hakkak'].sum():>3}")
print(f"  out-of-reference:   {(~root_compare['in_hakkak']).sum():>3}")

### 3.3) Peek at the reference hubs

Before running the formal enrichment test, look at the reference
hubs directly. The point of this peek is descriptive: do the symbols
read like canonical stress regulators (DREB2A, NAC019, ATAF1,
WRKY8…), or do they look unfamiliar? A list of recognizable names
is a sanity check that the upstream pipeline is producing
biologically plausible output.

Reference hubs that show up across both tissues — the "central in
both" candidates — get their own treatment in Section 5.

In [ ]:
display_cols = ['agi_id', 'symbol', 'family', 'group', 'n_hub_probes']

print("Shoot reference hubs:")
print(
    shoot_compare[shoot_compare['in_hakkak']][display_cols]
    .to_string(index=False)
)
print()
print("Root reference hubs:")
print(
    root_compare[root_compare['in_hakkak']][display_cols]
    .to_string(index=False)
)

### 3.4) Per-family breakdown

The per-family breakdown asks: of the reference hubs in each tissue,
which TF families are they from? Hakkak's set has 15 families with
sizes from 1 to 9; if any single family dominates the hub set
(say, WRKY landing in shoot or NAC landing in root), that's a
finding worth carrying into the synthesis.

The chart shows shoot and root side-by-side per family. Bars are
labeled with the count above each bar. Families are ordered by
total hub count (shoot + root) descending — the families with the
most hub representation across both tissues appear on the left.

In [ ]:
import matplotlib.pyplot as plt

# Per-family count of reference hubs, per tissue.
def _family_counts(compare_df):
    return (
        compare_df[compare_df['in_hakkak']]
        .groupby('family')
        .size()
    )

shoot_fam = _family_counts(shoot_compare)
root_fam  = _family_counts(root_compare)

# All families with at least one hub in either tissue, ordered by
# combined count (descending).
all_families = sorted(
    set(shoot_fam.index) | set(root_fam.index),
    key=lambda f: -(shoot_fam.get(f, 0) + root_fam.get(f, 0))
)
shoot_vals = [shoot_fam.get(f, 0) for f in all_families]
root_vals  = [root_fam.get(f, 0)  for f in all_families]

# Print the table first so the numbers are easy to copy into the writeup.
print(f"{'Family':<10}  {'Shoot':>6}  {'Root':>6}")
print("-" * 26)
for f, s, r in zip(all_families, shoot_vals, root_vals):
    print(f"{f:<10}  {s:>6}  {r:>6}")
print()

# Side-by-side bar chart.
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(all_families))
width = 0.4
bars_s = ax.bar([i - width/2 for i in x], shoot_vals, width, label='Shoot')
bars_r = ax.bar([i + width/2 for i in x], root_vals,  width, label='Root')
ax.bar_label(bars_s, padding=2, fontsize=9)
ax.bar_label(bars_r, padding=2, fontsize=9)
ax.set_xticks(list(x))
ax.set_xticklabels(all_families, rotation=45, ha='right')
ax.set_ylabel('Reference hubs (count)')
ax.set_title('Reference hubs by TF family, per tissue')
ax.legend()
plt.tight_layout()
plt.show()

The descriptive overlap is in hand: two comparison tables, two
counts, one per-family chart. The numbers say something about how
much overlap exists, but not whether the overlap is "more than
chance."

That's Section 4's job. Using the universe established in Section 2
and the hub counts from this section, Section 4 runs the
Bonferroni-corrected hypergeometric test the Week 1 pre-commit
locked in.

## 4) Test whether the overlap is more than chance

Section 3's numbers — 4 reference hubs in shoot, 26 in root — could
be explained two ways:

1. Hubs are genuinely enriched for known stress-regulating
   transcription factors (the prediction).
2. The hub set is large enough that some reference TFs will land in
   it by chance, even with no biological preference.

The hypergeometric test distinguishes these. Pre-committed in Week 1
as the test for this question, one-sided (right tail — we're asking
about enrichment, not depletion), one test per tissue, Bonferroni-
corrected for two tests so the family-wise error rate stays at 0.05.

The section has three sub-steps:

- **4.1** Build the 2×2 contingency table for each tissue. Confirms
  the numbers add up before the test runs.
- **4.2** Run the hypergeometric test for each tissue. Reports the
  observed count, the expected count under the null, the fold
  enrichment (effect size), and the p-value.
- **4.3** Apply Bonferroni correction and state the verdict per
  tissue at α = 0.025.

The fold enrichment matters as much as the p-value: a small p with a
1.05× fold is statistically interesting but biologically thin; a
large fold with a borderline p says the direction is real even if
the sample size isn't large enough to clear the threshold.

### 4.1) Build the 2×2 contingency tables

The 2×2 layout you locked in Week 1, with concrete numbers from
Sections 2 and 3:

|             | In Hakkak | Not in Hakkak | Total |
|-------------|-----------|---------------|-------|
| Hub         | a         | b             | a+b = n (hub set size) |
| Not hub     | c         | d             | c+d = N − n |
| **Total**   | a+c = K   | b+d = N − K   | **N (universe)** |

- `N` = the universe = `len(<tissue>_background_agi)` from Section 2.
- `K` = Hakkak AGIs in the universe = `len(hakkak_agi & <tissue>_background_agi)`.
- `n` = hub set size = `len(<tissue>_hubs_agi)`.
- `k` = observed successes = `len(<tissue>_hubs_agi & hakkak_agi)` = the
  reference hub count from Section 3.

The cell below computes all four values per tissue and prints the
contingency tables so the numbers are inspectable before the test
runs.

In [ ]:
def _table_params(hubs_agi, background_agi, reference_agi):
    """Return (N, K, n, k) for the hypergeometric test."""
    N = len(background_agi)
    K = len(reference_agi & background_agi)
    n = len(hubs_agi)
    k = len(hubs_agi & reference_agi)
    return N, K, n, k


def _print_2x2(tissue, N, K, n, k):
    """Print the 2x2 contingency table for one tissue."""
    table = pd.DataFrame(
        {
            'In Hakkak':     [k,        K - k,             K],
            'Not in Hakkak': [n - k,    N - n - K + k,     N - K],
            'Total':         [n,        N - n,             N],
        },
        index=['Hub', 'Not hub', 'Total'],
    )
    print(f"{tissue.capitalize()} contingency table:")
    print(table.to_string())
    print()


shoot_N, shoot_K, shoot_n, shoot_k = _table_params(
    shoot_hubs_agi, shoot_background_agi, hakkak_agi
)
root_N, root_K, root_n, root_k = _table_params(
    root_hubs_agi, root_background_agi, hakkak_agi
)

_print_2x2("shoot", shoot_N, shoot_K, shoot_n, shoot_k)
_print_2x2("root",  root_N,  root_K,  root_n,  root_k)

### 4.2) Run the hypergeometric test

`scipy.stats.hypergeom.sf(k-1, M, n, N)` returns the right-tail
probability `P(X ≥ k)` — the chance of seeing at least as many
reference hubs as we observed, if the null is true (hubs drawn at
random from the universe with no biological preference).

A naming note that catches people: scipy's hypergeom parameters use
different names than the textbook 2×2 convention. The mapping:

- scipy's `M` = our `N` (universe size).
- scipy's `n` = our `K` (number of successes in universe).
- scipy's `N` = our `n` (number of draws).

The `k - 1` offset matters too: `sf(k - 1, ...)` gives `P(X ≥ k)`,
which is what enrichment asks. `sf(k, ...)` would give `P(X > k)` —
off by one. `cdf(k, ...)` would give `P(X ≤ k)` — the depletion
direction, opposite of what we want.

Alongside the p-value, the cell reports the **fold enrichment**:
the observed proportion of reference hubs in the hub set, divided
by the expected proportion under the null. A fold of 1.0 means
"exactly as many as you'd expect"; 2.0 means twice as many; 5.0
means five times as many.

In [ ]:
from scipy.stats import hypergeom


def _run_test(tissue, N, K, n, k):
    # One-sided p-value: P(X >= k), right tail.
    # scipy's signature: hypergeom.sf(k_minus_1, population_size, successes_in_population, n_draws)
    p_value = hypergeom.sf(k - 1, N, K, n)

    # Expected count under the null.
    expected_k = n * K / N

    # Fold enrichment: observed proportion / expected proportion.
    observed_prop = k / n
    expected_prop = K / N
    fold = observed_prop / expected_prop if expected_prop > 0 else float('inf')

    return {
        'N': N, 'K': K, 'n': n, 'k': k,
        'expected_k': expected_k,
        'fold_enrichment': fold,
        'p_value': p_value,
    }


shoot_result = _run_test('shoot', shoot_N, shoot_K, shoot_n, shoot_k)
root_result  = _run_test('root',  root_N,  root_K,  root_n,  root_k)


def _print_test_row(tissue, r):
    print(f"  {tissue:<6}  "
          f"observed={r['k']:>3}   "
          f"expected={r['expected_k']:>5.2f}   "
          f"fold={r['fold_enrichment']:>5.2f}×   "
          f"p={r['p_value']:.3e}")


print("Hypergeometric test, one-sided (enrichment direction):")
print()
print(f"  {'tissue':<6}  {'observed':<11}  {'expected':<13}  {'fold':<10}  p-value")
print(f"  {'------':<6}  {'--------':<11}  {'--------':<13}  {'----':<10}  -------")
_print_test_row("shoot", shoot_result)
_print_test_row("root",  root_result)

### 4.3) Apply Bonferroni and state the verdict

Two tests, family-wise α = 0.05, Bonferroni correction → per-test
α = 0.025. A tissue's enrichment is reported as statistically
significant only if its p-value is below 0.025.

Bonferroni is conservative, especially with only two tests where
the alternatives (FDR-style corrections) would give slightly more
permissive thresholds. The conservatism is the price of simplicity;
for two tests it's small enough to live with.

A tissue that doesn't clear Bonferroni isn't proven to have no
enrichment — it's just that the data we have doesn't rise to the
threshold. The fold enrichment from Section 4.2 still describes
the direction and magnitude of the observed effect, and is the
honest summary in that case.

In [ ]:
ALPHA_FAMILY = 0.05
N_TESTS = 2
ALPHA_PER_TEST = ALPHA_FAMILY / N_TESTS  # 0.025

print(f"Family-wise α: {ALPHA_FAMILY}")
print(f"Number of tests: {N_TESTS}")
print(f"Per-test α (Bonferroni): {ALPHA_PER_TEST}")
print()


def _verdict(tissue, result, alpha):
    passes = result['p_value'] < alpha
    result['passes_bonferroni'] = bool(passes)
    result['alpha_per_test'] = alpha
    status = "PASSES" if passes else "DOES NOT PASS"
    print(f"  {tissue:<6}  p = {result['p_value']:.3e}  "
          f"vs α = {alpha}   →   {status}")


print("Verdict at Bonferroni-corrected α = 0.025:")
_verdict("Shoot", shoot_result, ALPHA_PER_TEST)
_verdict("Root",  root_result,  ALPHA_PER_TEST)

# Bundle for Section 6 to consume.
test_results = {
    'shoot': shoot_result,
    'root':  root_result,
    'alpha_family_wise': ALPHA_FAMILY,
    'n_tests': N_TESTS,
    'alpha_per_test': ALPHA_PER_TEST,
}

The formal test result is in hand for each tissue. Section 5 takes
a separate descriptive look at the 66 hubs shared between shoot
and root — the cross-tissue set N2 flagged as a tractable input
for closer inspection. Section 6 then pulls everything together
into the answer the question deserves.

## 5) Cross-tissue hubs

Sections 3 and 4 looked at each tissue separately. Section 5 takes
a different cut: the AGIs that surface as hubs in *both* networks.

N2's Section 6 preview reported 66 cross-tissue hub probes; after
the AGI mapping in Section 2, the count below collapses some
multi-probe-to-same-AGI cases and may be slightly smaller. These
are the genes that act as central in both shoot and root
co-expression structure — arguably the most biologically robust
candidates the pipeline surfaced, since both independently
constructed networks agreed on their centrality.

This isn't a third statistical test. The prediction locked in
Week 1 didn't include cross-tissue hubs, and adding a third
Bonferroni penalty for an analysis cut we didn't pre-commit would
be statistical malpractice. The point of this section is
descriptive: who are the cross-tissue hubs, and what do they look
like against the reference?

The section has two pieces:

- **5.1** Build a cross-tissue comparison table parallel to the
  per-tissue tables from Section 3.
- **5.2** Look at `249264_s_at` specifically — N2 flagged it as
  the single standout cross-tissue probe (mean kME 0.942, ranked
  in the top two for both tissues), explicitly handed off as a
  candidate worth following up on.

### 5.1) Build the cross-tissue comparison table

Set intersection on the two tissue hub sets gives the cross-tissue
AGIs. Joining against the deduplicated Hakkak reference produces
the comparison table, sorted reference-hubs-first the same way the
per-tissue tables are.

In [ ]:
# AGIs that appear as hubs in both tissues.
cross_tissue_agi = shoot_hubs_agi & root_hubs_agi

# Build the comparison table.
annotations = (
    hakkak_tf_dedup
    .rename(columns={
        'ORF': 'agi_id',
        'Symbol': 'symbol',
        'Family': 'family',
        'Group': 'group',
    })
    [['agi_id', 'symbol', 'family', 'group']]
)

cross_tissue_compare = (
    pd.DataFrame({'agi_id': sorted(cross_tissue_agi)})
    .merge(annotations, on='agi_id', how='left')
)
cross_tissue_compare['in_hakkak'] = cross_tissue_compare['family'].notna()
cross_tissue_compare = (
    cross_tissue_compare
    .sort_values(['in_hakkak', 'agi_id'], ascending=[False, True])
    .reset_index(drop=True)
)

# Report.
n_total = len(cross_tissue_compare)
n_in = cross_tissue_compare['in_hakkak'].sum()
n_out = n_total - n_in

print(f"Cross-tissue hub AGIs: {n_total}")
print(f"  reference hubs:     {n_in:>3}  ({n_in / n_total * 100:.1f}% of cross-tissue)")
print(f"  out-of-reference:   {n_out:>3}  ({n_out / n_total * 100:.1f}% of cross-tissue)")
print()

if n_in > 0:
    print("Cross-tissue reference hubs:")
    print(
        cross_tissue_compare[cross_tissue_compare['in_hakkak']]
        [['agi_id', 'symbol', 'family', 'group']]
        .to_string(index=False)
    )
else:
    print("No cross-tissue hubs are in the reference set.")

### 5.2) The N2 standout

N2's cross-tissue preview surfaced one probe as the strongest
"central in both tissues" candidate: `249264_s_at`, with a mean kME
of 0.942 averaged across tissues, ranked first in shoot and second
in root within its respective modules.

Whether that probe maps to a known reference TF or to an
out-of-reference gene is exactly the kind of question N2's
handoff invited Section 5 to answer. If it's a reference TF, the
prediction's "familiar names" claim gets its strongest single piece
of support. If it's out-of-reference, this is the single most
defensible novel candidate the pipeline has produced — central in
both tissues, kME above 0.94, supported by independent network
construction.

In [ ]:
STANDOUT_PROBE = '249264_s_at'
standout_agi = probe_to_agi.get(STANDOUT_PROBE)

print(f"N2 standout probe:  {STANDOUT_PROBE}")
print(f"AGI mapping:        {standout_agi}")
print()

if standout_agi is None:
    print(f"Surprise: {STANDOUT_PROBE} has no AGI mapping in GPL198.")
elif standout_agi not in cross_tissue_agi:
    print(f"Surprise: {standout_agi} is not in the cross-tissue set "
          f"this section computed. Worth checking why.")
else:
    standout_row = cross_tissue_compare[
        cross_tissue_compare['agi_id'] == standout_agi
    ].iloc[0]

    if standout_row['in_hakkak']:
        print(f"{standout_agi} is a reference cross-tissue hub.")
        print(f"  Symbol:  {standout_row['symbol']}")
        print(f"  Family:  {standout_row['family']}")
        print(f"  Group:   {standout_row['group']}")
        print()
        print("This is the prediction's 'familiar names' claim landing on "
              "the single most central gene the pipeline surfaced.")
    else:
        print(f"{standout_agi} is an out-of-reference cross-tissue hub.")
        print()
        print("Central in both networks (mean kME 0.942), confirmed by "
              "independent network construction, and not in the curated "
              "reference. The strongest single candidate the pipeline "
              "surfaced for follow-up.")

The cross-tissue cut sharpens the descriptive picture but doesn't
change the formal answer from Section 4. With the per-tissue test
results and the cross-tissue context both in hand, Section 6
pulls everything together: the headline numbers, the asymmetry
between tissues, the reference hubs that recapitulate canonical
biology, the out-of-reference candidates worth following up on,
and the honest answer to R1-Q2's question.

## 6) What we learned

R1-Q2 asked whether hub genes identified through co-expression
analysis on AtGenExpress are enriched for known stress regulators
relative to non-hub genes. Sections 3 through 5 produced the
descriptive overlap, the formal test, and the cross-tissue cut.
This section pulls them together: the headline, the reference
hubs the pipeline recovered, the out-of-reference candidates it
surfaced, and the saved artifacts that drive the writeup.

### 6.1) The headline

The answer to R1-Q2 is asymmetric across tissues.

In **root**, the prediction lands cleanly. Of 430 unique hub
AGIs, 26 are in Hakkak's reference set of stress-regulating
transcription factors — roughly six times what chance alone
would produce. The hypergeometric test returns p = 3.83 × 10⁻¹⁸,
well below the Bonferroni-corrected threshold of α = 0.025.
Hubs identified through co-expression analysis on root tissue
are enriched for known stress regulators.

In **shoot**, the same prediction shows only a directional
trend. Of 210 unique hub AGIs, 4 are in the reference set —
about twice what chance would produce. The hypergeometric test
returns p = 0.119, which does not clear Bonferroni's
α = 0.025. The direction of the effect is real, but with only
4 observed reference hubs in a hub set of 210, the test does
not have power to distinguish the result from chance.

The asymmetry is consistent with the experimental design's
biological geometry. AtGenExpress's stresses — drought, salt,
osmotic, cold, heat — first challenge water and ion balance at
the root before any systemic signal reaches shoot tissue. Root
is where the regulatory signal concentrates in this dataset.

### 6.2) The reference hubs the pipeline recovered

The reference hubs are the genes the prediction expected to find —
the canonical stress regulators the student would recognize. The
pipeline recovered:

- **In shoot:** 4 genes — NF-YA5 (NF-YA), ATHB-12 (HD-ZIP),
  NAC072 (NAC), and a bZIP family member.
- **In root:** 26 genes spanning 8 families. NAC (6 hubs) is the
  most represented; MYB (5), WRKY (4), and C2H2 (4) follow.
  ERF (3), HD-ZIP (2), NF-YA (1), and HSF (1) round out the
  set. The families most prominent in Hakkak's reference (NAC,
  MYB, WRKY) are also the families most strongly represented
  in the root hub set.

Three of these are recovered in **both** tissues — the strongest
individual confirmations of the prediction, since both
independently constructed networks agreed on their centrality:

- **NF-YA5** (AT1G54160) — Hakkak's only NF-YA family member.
  An ABA-responsive nuclear factor with established roles in
  drought tolerance.
- **ATHB-12** (AT3G61890) — HD-ZIP family. Drought- and
  ABA-induced; the canonical example of an HD-ZIP class I
  transcription factor in stress contexts.
- **NAC072** (AT4G27410) — NAC family. Also referred to as
  RD26 in parts of the literature. One of the canonical NAC TFs
  in the abscisic acid / drought response.

All three are up-regulated in Hakkak's analysis, all three are
in ABA / drought-adjacent pathways, and all three show up as
central in two independently constructed co-expression networks.

Notable absence worth flagging for the writeup: **bHLH**.
Hakkak's reference has 7 bHLH transcription factors — the
second-largest family in the set. Zero bHLH hubs surface in
either tissue. A useful pointer to where the data-driven approach
diverges from the canonical view.

### 6.3) The out-of-reference hubs and the standout candidate

The prediction also said the overlap would be "substantial but
not complete" — that some hubs would be familiar names and
others wouldn't be. That part holds clearly: the overwhelming
majority of hub AGIs are not in the curated reference (about
98% of shoot hubs, 94% of root hubs).

These out-of-reference hubs are the candidates the data-driven
approach surfaces beyond what prior knowledge alone would have
predicted. One caveat to carry forward: "out-of-reference" does
not necessarily mean "novel." The curated reference is bounded
by Hakkak's 60-gene TF table, not by all known stress
regulators. Some of these out-of-reference hubs may be
well-characterized stress genes that simply aren't in this
particular reference; others may be genuinely undescribed.
Distinguishing the two is beyond the scope of this notebook.

The strongest single candidate comes from the cross-tissue cut.
Of the 66 hub AGIs that appear in both networks, 63 are
out-of-reference. The probe `249264_s_at` — AGI **AT5G41740** —
was flagged by N2 as the standout cross-tissue probe, with a
mean kME of 0.942 across tissues and a rank of first in shoot
and second in root within its respective modules. Section 5
confirmed that this gene is not in Hakkak's reference.

AT5G41740 has the strongest support the pipeline can produce
for a candidate: high centrality in two independently
constructed networks, surfaced by the kME ≥ 0.8 threshold N2
locked in Week 1, and arrived at without any prior-knowledge
bias. The writeup should characterize it directly — look up the
gene's annotation in TAIR, check whether it has been described
in stress-response literature, and discuss its candidacy on its
merits.

### 6.4) Save the paper-ready artifacts

Four files drop into the R1-Q2 output directory for the writeup
and presentation:

- `shoot_compare.parquet` — per-AGI comparison table for shoot
  hubs, one row per unique hub AGI.
- `root_compare.parquet` — same shape for root hubs.
- `cross_tissue_compare.parquet` — same shape for the 66
  cross-tissue hub AGIs.
- `comparison_summary.json` — the headline numbers in serialized
  form: per-tissue test results, the three shared reference
  hubs, and the standout candidate's identity. The structure is
  designed to be copy-paste-friendly for the writeup.

In [ ]:
### 6.4) Save the paper-ready artifacts (code)

import json

# ---------------- Three per-AGI parquets ----------------

shoot_compare_path = OUTPUT_DIR / "shoot_compare.parquet"
root_compare_path  = OUTPUT_DIR / "root_compare.parquet"
cross_tissue_path  = OUTPUT_DIR / "cross_tissue_compare.parquet"

shoot_compare.to_parquet(shoot_compare_path)
root_compare.to_parquet(root_compare_path)
cross_tissue_compare.to_parquet(cross_tissue_path)

# Round-trip verify: row counts and columns survive save/load.
for df, path in [
    (shoot_compare,        shoot_compare_path),
    (root_compare,         root_compare_path),
    (cross_tissue_compare, cross_tissue_path),
]:
    reloaded = pd.read_parquet(path)
    assert len(reloaded) == len(df), f"{path.name}: row-count mismatch on round-trip"
    assert list(reloaded.columns) == list(df.columns), (
        f"{path.name}: column mismatch on round-trip"
    )

# ---------------- comparison_summary.json ----------------
#
# Three top-level keys, mapping directly onto the Section 6.4 description:
#   - test_results:          per-tissue test results plus alpha bookkeeping
#   - shared_reference_hubs: cross-tissue hubs that are in Hakkak's reference
#   - standout_candidate:    N2's standout probe's identity in this analysis
#
# Contents are derived from the in-scope variables rather than from the
# Section 6 prose, so the file always reflects what was actually computed.

# Shared reference hubs: the rows of cross_tissue_compare where in_hakkak is True.
# Pandas NaNs (e.g., a missing Symbol from the Hakkak file) get coerced to None
# at construction time so the JSON output is clean.
shared_reference_hubs = []
for _, row in cross_tissue_compare[cross_tissue_compare['in_hakkak']].iterrows():
    shared_reference_hubs.append({
        'agi_id': row['agi_id'],
        'symbol': None if pd.isna(row['symbol']) else row['symbol'],
        'family': None if pd.isna(row['family']) else row['family'],
        'group':  None if pd.isna(row['group'])  else row['group'],
    })

# Standout candidate: three cases, mirroring Cell 42's branching logic.
if standout_agi is None:
    # The probe has no AGI mapping in GPL198 — the surprise case from Cell 42.
    standout_block = {
        "probe_id":        STANDOUT_PROBE,
        "agi_id":          None,
        "in_cross_tissue": False,
        "in_reference":    None,
        "symbol":          None,
        "family":          None,
        "group":           None,
    }
elif standout_agi not in cross_tissue_agi:
    # The AGI exists but didn't land in the cross-tissue set this section
    # computed — also a flagged surprise in Cell 42.
    standout_block = {
        "probe_id":        STANDOUT_PROBE,
        "agi_id":          standout_agi,
        "in_cross_tissue": False,
        "in_reference":    None,
        "symbol":          None,
        "family":          None,
        "group":           None,
    }
else:
    # The expected case: standout is in the cross-tissue set. Look up
    # its row and report whether it's in or out of the reference.
    row = cross_tissue_compare[cross_tissue_compare['agi_id'] == standout_agi].iloc[0]
    standout_block = {
        "probe_id":        STANDOUT_PROBE,
        "agi_id":          standout_agi,
        "in_cross_tissue": True,
        "in_reference":    bool(row['in_hakkak']),
        "symbol":          None if pd.isna(row['symbol']) else row['symbol'],
        "family":          None if pd.isna(row['family']) else row['family'],
        "group":           None if pd.isna(row['group'])  else row['group'],
    }

comparison_summary = {
    "test_results":          test_results,
    "shared_reference_hubs": shared_reference_hubs,
    "standout_candidate":    standout_block,
}

# scipy.stats.hypergeom returns numpy floats; json's default encoder
# doesn't handle those. The .item() method on numpy scalars returns the
# native Python equivalent and works as a default= hook.
def _json_default(obj):
    try:
        return obj.item()
    except AttributeError:
        raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

summary_path = OUTPUT_DIR / "comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(comparison_summary, f, indent=2, default=_json_default)

# Round-trip verify the JSON: top-level keys and test_results keys
# must survive a save/load cycle.
with open(summary_path) as f:
    reloaded = json.load(f)
assert set(reloaded.keys()) == set(comparison_summary.keys()), (
    "comparison_summary.json: top-level keys mismatch on round-trip"
)
assert set(reloaded['test_results'].keys()) == set(test_results.keys()), (
    "comparison_summary.json: test_results keys mismatch on round-trip"
)

# ---------------- Print-back ----------------

print("Saved artifacts:")
for path in [shoot_compare_path, root_compare_path, cross_tissue_path, summary_path]:
    print(f"  {path.name:<32}  {path.stat().st_size / 1e3:>6.1f} KB")
print()
print("comparison_summary.json contents:")
print(f"  test_results:           per-tissue stats for shoot and root")
print(f"  shared_reference_hubs:  {len(shared_reference_hubs)} entries")
agi_display = standout_block['agi_id'] or '(no AGI mapping)'
print(f"  standout_candidate:     {STANDOUT_PROBE} → {agi_display}")

### 6.5) Closing

R1-Q2 asked whether genes identified as hubs through WGCNA-style
co-expression analysis on AtGenExpress are enriched for known
stress regulators relative to non-hub genes. The answer is yes
for root, with a strong fold enrichment and a p-value well
below the multiple-testing-corrected threshold; it is not
formally yes for shoot, although the direction of the effect is
positive and the magnitude is roughly twofold. The tissue
asymmetry is consistent with where AtGenExpress's stresses
register physiologically. The cross-tissue subset surfaces three
canonical regulators (NF-YA5, ATHB-12, NAC072) plus 63
out-of-reference candidates, with AT5G41740 standing out as the
strongest individual candidate worth follow-up.

R1-Q2 is analytically complete. The paper and presentation are
external to the notebook chain.